# Notebook 1: Fundamentals of JAX and Simply Utilities

Welcome to the first notebook of the Simply curriculum! In this notebook, you will learn the fundamentally functional style of JAX programming and see how Simply extends it to manage parameters across large-scale model architectures.

First, let's configure the environment to securely access the `simply` module located in our `third-party/simply` directory alongside this notebook.

In [20]:
import sys
import os

# Safely add the simply path to system modules so that we can import them within the notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '.', 'third-party', 'simply'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax
import jax.numpy as jnp
import numpy as np
print(f"JAX version : {jax.__version__}")

JAX version : 0.9.2


## 1. JIT Compilation (`jax.jit`)
JAX uses XLA (Accelerated Linear Algebra) to compile functions directly to GPU/TPU or optimized CPU instructions. You apply JIT using `jax.jit`.

### Exercise 1: JIT-compile a simple mathematical operation.
Create a function `scaling_activation` that returns $x + \tanh(x)$. Then trace it via JIT and benchmark it vs non-compiled execution.

In [7]:
# Exercise 1 Code

def scaling_activation(x):
    # TODO 1: Implement the function x + jnp.tanh(x)
    return x + jnp.tanh(x)

# TODO 2: Wrap the function with jax.jit
fast_scaling_activation = jax.jit(scaling_activation)

# Let's test it out
x_dummy = jnp.linspace(-3, 3, 1000)
if fast_scaling_activation is not None:
    # Compiles on first run!
    res = fast_scaling_activation(x_dummy)
    print(f"res.shape = {res.shape}")
    print("JIT function executed successfully!")

res.shape = (1000,)
JIT function executed successfully!


## 2. Automatic Vectorization (`jax.vmap`)
In JAX, we prefer to write logic mapping to a *single* sample or entity, and then use `jax.vmap` to automatically batch the code. This improves readability significantly.

### Exercise 2: Vmap batching
You have a complex function `dot_and_add` scaling two 1D vectors and returning a scalar. Your goal is to apply it to a batch of vectors simultaneously.

In [8]:
?jax.vmap

Signature:
jax.vmap(
    fun: 'F',
    in_axes: 'int | None | Sequence[Any]' = 0,
    out_axes: 'Any' = 0,
    axis_name: 'AxisName | None' = None,
    axis_size: 'int | None' = None,
    spmd_axis_name: 'AxisName | tuple[AxisName, ...] | None' = None,
    sum_match: 'bool' = False,
) -> 'F'
Docstring:
Vectorizing map. Creates a function which maps ``fun`` over argument axes.

Args:
  fun: Function to be mapped over additional axes.
  in_axes: An integer, None, or sequence of values specifying which input
    array axes to map over.

    If each positional argument to ``fun`` is an array, then ``in_axes`` can
    be an integer, a None, or a tuple of integers and Nones with length equal
    to the number of positional arguments to ``fun``. An integer or ``None``
    indicates which array axis to map over for all arguments (with ``None``
    indicating not to map any axis), and a tuple indicates which axis to map
    for each corresponding positional argument. Axis integers must be in th

In [9]:
# Exercise 2 Code

def dot_and_add(vec_a, vec_b):
    # A function meant only for 1D arrays!
    return jnp.dot(vec_a, vec_b) + 5.0

# Let's create a batch of 10 vectors of size 5
batch_a = jnp.ones((10, 5))
batch_b = jnp.ones((10, 5))

# TODO: Use jax.vmap to batch `dot_and_add` and apply it to `batch_a` and `batch_b`.
# Vmapped function should return an array of shape (10,)
batched_dot_and_add = jax.vmap(dot_and_add, in_axes=(0, 0))

if batched_dot_and_add is not None:
    res = batched_dot_and_add(batch_a, batch_b)
    print(f"Batched result shape: {res.shape}")

Batched result shape: (10,)


## 3. Gradients (`jax.grad`)
Derivatives are a primary citizen of JAX. Using `jax.grad`, JAX automatically reverses paths to compute gradients to your loss functions. To receive both the raw loss and the gradient, use `jax.value_and_grad`.

### Exercise 3: Gradient extraction
Write a dummy loss function that computes the sum of squares, and compute its gradient with respect to its inputs.

In [10]:
?jax.value_and_grad

Signature:
jax.value_and_grad(
    fun: 'Callable',
    argnums: 'int | Sequence[int]' = 0,
    has_aux: 'bool' = False,
    holomorphic: 'bool' = False,
    allow_int: 'bool' = False,
    reduce_axes: 'Sequence[AxisName]' = (),
) -> 'Callable[..., tuple[Any, Any]]'
Docstring:
Create a function that evaluates both ``fun`` and the gradient of ``fun``.

Args:
  fun: Function to be differentiated. Its arguments at positions specified by
    ``argnums`` should be arrays, scalars, or standard Python containers. It
    should return a scalar (which includes arrays with shape ``()`` but not
    arrays with shape ``(1,)`` etc.)
  argnums: Optional, integer or sequence of integers. Specifies which
    positional argument(s) to differentiate with respect to (default 0).
  has_aux: Optional, bool. Indicates whether ``fun`` returns a pair where the
    first element is considered the output of the mathematical function to be
    differentiated and the second element is auxiliary data. Default Fals

In [15]:
# Exercise 3 Code

def mock_loss_fn(params, inputs):
    # params: A single weight matrix, inputs: data.
    # Computes a mock MSE-like loss
    predictions = inputs @ params
    return jnp.mean((predictions - 0.5) ** 2)

# Initial params and inputs
initial_params = jnp.array([[0.1, -0.2], [0.5, 0.4]])
inputs = jnp.array([[1.0, 0.5], [-0.5, 1.0]])

# TODO: Calculate both the loss value and the gradient of mock_loss_fn w.r.t `params` (arg 0)
loss_val, param_grads = jax.value_and_grad(mock_loss_fn, argnums=0)(initial_params, inputs)

if loss_val is not None:
    print(f"Loss: {loss_val}")
    print(f"Gradients:\n{param_grads}")

Loss: 0.06875000149011612
Gradients:
[[-0.0625     -0.25      ]
 [-0.06250001 -0.125     ]]


## 4. PyTrees & Stateless Functional Programming
Large architectures like LLMs are represented as vast dictionaries of parameters. JAX natively operates on these nested structures using the PyTree abstraction (`jax.tree_util`). Note that Simply provides some nice utilities for these in `simply.utils.pytree`.

### Exercise 4: PyTree manipulation
Write a custom map function that filters out all arrays in the nested PyTree that have a sum smaller than 0. You'll use JAX's `tree_map` equivalent here.

In [17]:
# Exercise 4 Code

big_model_params = {
    'layer1': {'weights': jnp.array([1.0, 2.0]), 'bias': jnp.array([-1.0, -2.0, -3.0])},
    'layer2': {'scale': jnp.array([-0.5, 0.5]), 'offset': jnp.array([5.0])}
}

# TODO: Using `jax.tree_util.tree_map`, return a new PyTree where any leaf with a 
# sum less than 0 is replaced by jnp.zeros_like(leaf)
filtered_params = jax.tree_util.tree_map(lambda x: jnp.where(jnp.sum(x) < 0, jnp.zeros_like(x), x), big_model_params)

if filtered_params is not None:
    print(f"Filtered tree: {filtered_params}")

Filtered tree: {'layer1': {'bias': Array([0., 0., 0.], dtype=float32), 'weights': Array([1., 2.], dtype=float32)}, 'layer2': {'offset': Array([5.], dtype=float32), 'scale': Array([-0.5,  0.5], dtype=float32)}}


## 5. Simply Context: `AnnotatedArray`
When inspecting the Simply codebase, you'll constantly see `AnnotatedArray.create(x, dim_annotation='...')`. 
This conceptually tags a raw array with metadata needed by Simply to perform Mesh Sharding across multiple TPUs automatically.

Simply models wrap standard arrays in `AnnotatedArray` PyTrees.

### Exercise 5: Extracting and inspecting AnnotatedArrays
Create an AnnotatedArray representing parameter weights. Finally, extract it back into raw arrays using `simply.utils.common.get_raw_arrays`.

In [ ]:
  def test_annotated_array(self):
    x = common.AnnotatedArray.create(jnp.array([1, 2]), dim_annotation='io')
    self.assertEqual(x.array.tolist(), [1, 2])
    self.assertEqual(x.metadata, {'dim_annotation': 'io'})
    z = common.AnnotatedArray.create(jnp.array([5, 6]), dim_annotation='oi')
    y = jnp.array([3, 4])
    y1, z1 = common.transfer_metadata((x, x), (y, z))
    print('=' * 50)
    print(f'y1: {y1}')
    print('=' * 50)
    self.assertEqual(y1.metadata, {'dim_annotation': 'io'})
    self.assertEqual(z1.metadata, {'dim_annotation': 'io'})
    self.assertEqual(y1.array.tolist(), y.tolist())
    self.assertEqual(z1.array.tolist(), z.array.tolist())

--- Testing 'Char Mode' (No Spaces) ---
Array (128, 128) + 'io' -> ✅ EXACT MATCH (2 labels for 2 dims)
Array (128, 128) + '.io' -> ⚠️ PARTIAL/OVERFLOW (3 labels for 2 dims)

--- Testing 'Word Mode' (With Spaces) ---
Array (128, 128) + '. io' -> ✅ EXACT MATCH (2 labels for 2 dims)
Array (128, 128) + 'batch . io' -> ⚠️ PARTIAL/OVERFLOW (3 labels for 2 dims)

--- Testing 1D Arrays (Bias) ---
Array (128,) + 'o' -> ✅ EXACT MATCH (1 labels for 1 dims)
Array (128,) + '.o' -> ⚠️ PARTIAL/OVERFLOW (2 labels for 1 dims)
✅ 'io' is VALID for shape (128, 128)
✅ '.ioxy ix ixi mdmd  dkdnnl' is VALID for shape (128, 128)


In [36]:
# Exercise 5 Code
from simply.utils.common import AnnotatedArray, get_raw_arrays

# Create some raw jax parameters
raw_weight = jax.random.normal(jax.random.PRNGKey(0), (128, 128))

# TODO 1: Create an AnnotatedArray wrapping `raw_weight` with a `dim_annotation` string of '.io' (meaning input/output sharding).
annotated_weight = AnnotatedArray.create(raw_weight, dim_annotation='io')

# An imaginary LLM dictionary tree
llm_tree = {'ffn': annotated_weight, 'bias': jnp.zeros(128)}

# TODO 2: Call `get_raw_arrays` on `llm_tree` to strip out all AnnotatedArrays recursively, 
# returning just raw jax primitives.
stripped_tree = get_raw_arrays(llm_tree)

if stripped_tree is not None:
    print("Successfully stripped AnnotatedArray metadata!")
    print(f"Type of ffn branch: {type(stripped_tree['ffn'])}")

Successfully stripped AnnotatedArray metadata!
Type of ffn branch: <class 'jaxlib._jax.ArrayImpl'>


---

## ✨ Bonus Material: Additional Topics & Exercises

The following exercises and concepts were merged from the previous `01_jax_basics_and_design.ipynb` curriculum.

# 01: JAX Foundations & The Functional Software Pattern
Welcome to the first notebook in your journey to learning JAX, Large Language Model Architectures, and the `simply` codebase!

If your goal is to use `simply` for your own LLM experiments, you must first transition from an **Object-Oriented** mindset (common in PyTorch) to a **Functional** mindset (mandatory in JAX). Let's dive in.

In [ ]:
# Setup environment for imports
import sys
import os
sys.path.append(os.path.abspath('./third-party/simply'))

import jax
import jax.numpy as jnp

## 1. Pure Functions vs Mutating State
In an OOP framework like PyTorch, a layer often mutates its internal state when running a forward pass, or when gradients are applied.
`self.weight.grad = ...`

JAX requires **Pure Functions**. A pure function:
1. Cannot mutate its inputs or global state.
2. Given identical inputs, always produces identical outputs.

Because of this, neural networks in JAX (and in `simply`) are explicitly passed their parameters on every single forward pass.

## 2. JAX JIT (Just-In-Time Compilation)
JIT compiles your python functions down to XLA (Accelerated Linear Algebra) code that runs incredibly fast on CPU, GPU, or TPU.

In [ ]:
def relu(x):
    return jnp.maximum(0, x)

# JIT compile the function
fast_relu = jax.jit(relu)

x = jnp.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Standard ReLU output:", relu(x))
print("JIT ReLU output:", fast_relu(x))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Standard ReLU output: [0. 0. 0. 1. 2.]
JIT ReLU output: [0. 0. 0. 1. 2.]


In [ ]:
%%timeit

fast_relu(x)

3.75 μs ± 58 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
%%timeit

relu(x)

7.24 μs ± 228 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


## 3. The `SimplyModule` Abstraction
In `simply/utils/module.py`, you'll find `SimplyModule`, which acts as a structured way to separate the *definition* of a module from its *state*.
It has two core methods:
- `init(prng_key)`: Generates and returns a dictionary of randomly initialized parameters.
- `apply(params, x)`: Computes the forward pass using explicitly passed parameters.

This strictly conforms to JAX's pure function requirement!

In [ ]:
from simply.utils.module import SimplyModule, ModuleRegistry
from dataclasses import dataclass

@ModuleRegistry.register
@dataclass
class MySimpleLinear(SimplyModule):
    features: int

    def init(self, prng_key):
        # Return parameters for this module
        return {"weight": jax.random.normal(prng_key, (10, self.features))}
        
    def apply(self, params, x):
        # Notice how `params` is explicitly passed here!
        # State is NOT stored on `self`.
        return jnp.dot(x, params["weight"])

In [ ]:
# Let's test our simple module!
rng = jax.random.PRNGKey(42)
module = MySimpleLinear(features=5)

# 1. Initialize parameters
params = module.init(rng)
print("Initialized parameters keys:", params.keys())
print("Weight shape:", params["weight"].shape)

# 2. Run the forward pass
x_input = jnp.ones((2, 10))
output = module.apply(params, x_input)
print("Output shape:", output.shape)

Initialized parameters keys: dict_keys(['weight'])
Weight shape: (10, 5)
Output shape: (2, 5)


## Exercise: Fill-in-the-blank
Create an instance of `MySimpleLinear`, generate the parameters, and `vmap` (vectorizing map) the `apply` function over a newly shaped input of `(5, 2, 10)` representing `[batch, sequence, dim]`.

Remember `jax.vmap` automatically vectorizes pure functions over leading axes.

In [20]:
# 1. Instantiate module
my_module = MySimpleLinear(features=5)

# 2. Vectorized apply function using jax.vmap
# We sweep over the batch dimension of x, but the parameters remain the same!
vectorized_apply = jax.vmap(my_module.apply, in_axes=(None, 0))

test_input = jnp.ones((5, 2, 10))
out = vectorized_apply(params, test_input)
print(out.shape) # Should be (5, 2, 5)

(5, 2, 5)


### Solution

In [24]:
my_module = MySimpleLinear(features=5)

# in_axes specifies which axes of the arguments to map over.
# params dict is shared across the batch (None), x sweeps over dimension 0 (0)
vectorized_apply = jax.vmap(my_module.apply, in_axes=(None, 0))

test_input = jnp.ones((5, 2, 10))
# The 'params' variable was created in the cell above
out = vectorized_apply(params, test_input)
print("Vectorized output shape:", out.shape)

Vectorized output shape: (5, 2, 5)


## Exercise 2: Advanced `vmap` on multiple batched arguments
Now that you've seen `in_axes=(None, 0)`, let's try mapping over multiple arguments in different ways. 

Given the function `add_vectors(x, y)` which takes two 1D vectors and adds them, your goal is to use `vmap` to solve two problems:
1. Add a **batch** of vectors `X_batch` to another **batch** of vectors `Y_batch`.
2. Add a **batch** of vectors `X_batch` to a **single** vector `y_single`.

In [26]:
def add_vectors(x, y):
    return x + y

X_batch = jnp.ones((5, 3))  # 5 vectors of size 3
Y_batch = jnp.ones((5, 3)) * 2
y_single = jnp.array([10.0, 20.0, 30.0])

# TASK 1: Both inputs are batched.
vmapped_add_both = jax.vmap(add_vectors, in_axes=(0, 0))
res1 = vmapped_add_both(X_batch, Y_batch)
print("Task 1 Shape:", res1.shape)

# TASK 2: Only the first input is batched. The second is shared.
vmapped_add_shared = jax.vmap(add_vectors, in_axes=(0, None))
res2 = vmapped_add_shared(X_batch, y_single)
print("Task 2 Shape:", res2.shape)

Task 1 Shape: (5, 3)
Task 2 Shape: (5, 3)


### Solution 2

In [ ]:
# TASK 1: Sweep over dimension 0 of X_batch, and dimension 0 of Y_batch.
vmapped_add_both = jax.vmap(add_vectors, in_axes=(0, 0))
res1 = vmapped_add_both(X_batch, Y_batch)
print("Task 1 Shape:", res1.shape) # (5, 3)

# TASK 2: Sweep over dimension 0 of X_batch, but don't batch y_single.
vmapped_add_shared = jax.vmap(add_vectors, in_axes=(0, None))
res2 = vmapped_add_shared(X_batch, y_single)
print("Task 2 Shape:", res2.shape) # (5, 3)